# PS12 — Frame Interpolation on Colab (T4) + Drive persistence
Runtime → Change runtime type → **T4 GPU**. Drive holds data/weights/results so reruns are cheap.
Data is downloaded via `data_setup.py` **only if not already in Drive**.

In [ ]:
# 1) GPU + Drive (persistent storage)
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os
PERSIST = '/content/drive/MyDrive/ps12'
os.makedirs(PERSIST, exist_ok=True)
print('persistent root:', PERSIST)

In [ ]:
# 2) Config + credentials (prompted, not stored in the notebook)
import getpass, os
REPO_URL = 'https://github.com/<you>/ps12.git'  #@param {type:'string'}
os.environ['MOSDAC_USERNAME'] = input('MOSDAC username (blank to skip INSAT): ').strip()
if os.environ['MOSDAC_USERNAME']:
    os.environ['MOSDAC_PASSWORD'] = getpass.getpass('MOSDAC password: ')
hf = getpass.getpass('HF_TOKEN (optional, Enter to skip): ')
if hf: os.environ['HF_TOKEN'] = hf

In [ ]:
# 3) Get the code + deps + model repos
%cd /content
![ -d ps12 ] || git clone $REPO_URL ps12
%cd /content/ps12
!pip -q install -r requirements-local.txt
!python data_setup.py --clone all

In [ ]:
# 4) Link Drive-persistent dirs, then run data_setup ONLY if data is missing
import os, pathlib
for d in ['data', 'weights', 'validation_report', 'outputs']:
    src = f'{PERSIST}/{d}'; os.makedirs(src, exist_ok=True)
    lnk = f'/content/ps12/{d}'
    if os.path.islink(lnk) or not os.path.exists(lnk):
        if os.path.exists(lnk): import shutil; shutil.rmtree(lnk, ignore_errors=True)
        os.symlink(src, lnk)
have = any(pathlib.Path('data').glob('*/*')) or any(pathlib.Path('samples').glob('*/*'))
if not have:
    !python data_setup.py --download goes --start 2025-10-01 --end 2025-10-02 --max-gb 20
    !python data_setup.py --download insat --max-gb 15
    !python data_setup.py --build-index --source goes19 --step-min 10
else:
    print('data already present in Drive — skipping download')

In [ ]:
# 5) Deterministic battery (gate), then train the custom model + fine-tune RIFE
!pytest -q tests/
!python -m src.train.finetune --index data/index/goes19_triplets.json --steps 20000 --out weights/unet
# optional: fine-tune existing RIFE (needs weights/rife). 
# !python -m src.train.rife_finetune --index data/index/goes19_triplets.json --out weights/rife_ft --steps 15000

In [ ]:
# 6) Self-supervised INSAT adaptation + validation report (committed-style, saved to Drive)
!python -m src.train.insat_selfsup --index data/index/insat3dr_triplets.json --init weights/unet/best.pt --out weights/unet_insat --steps 5000 || echo 'INSAT index not built yet'
import json
from src.eval.report import run_eval
idx = json.load(open('data/index/goes19_triplets.json'))
run_eval(idx['triplets'], 'goes19', ['classical','raft','unet','rife_ft'], 'validation_report/goes19', max_triplets=20)
print('Results saved to Drive at', PERSIST + '/validation_report')

In [ ]:
# 7) Drive the dashboard from YOUR PC browser (UI runs on Colab's T4, tunnelled via ngrok)
import getpass, os, time
os.environ['NGROK_AUTHTOKEN'] = getpass.getpass('ngrok authtoken (free at dashboard.ngrok.com): ')
!pip -q install streamlit pyngrok
get_ipython().system_raw('python cloud/serve_dashboard.py > dash.log 2>&1 &')
time.sleep(15)
!grep -A1 'Open the dashboard' dash.log || tail -25 dash.log
# -> click the printed https URL to use the dashboard locally while compute stays on the T4